# TabFM (Google) — Regression

Demonstrates **TabFM** on the shared retail/CPG regression datasets. Self-hosted
weights, single forward-pass inference.

> **License note:** TabFM is **non-commercial**. Evaluation only. See [`../README.md`](../README.md).

**Compute:** GPU cluster recommended.

**Prerequisite:** run [`shared/notebooks/00_data_preparation.ipynb`](../../../shared/notebooks/00_data_preparation.ipynb) first.


In [ ]:
%pip install tabfm[pytorch] scikit-learn pandas matplotlib mlflow --quiet

In [ ]:
dbutils.library.restartPython()

## Configuration


In [ ]:
import os, sys

CATALOG = "tabular_fm"
SCHEMA = "default"
spark.sql(f"USE CATALOG {CATALOG}")
spark.sql(f"USE SCHEMA {SCHEMA}")

current_user = spark.sql("SELECT current_user()").collect()[0][0]
MLFLOW_EXPERIMENT_NAME = f"/Users/{current_user}/tabular-fm-databricks"

REPO_ROOT = os.path.abspath(os.path.join(os.getcwd(), "..", "..", ".."))
COMMON_PATH = os.path.join(REPO_ROOT, "common")
if COMMON_PATH not in sys.path:
    sys.path.insert(0, COMMON_PATH)

print(f"Catalog/schema: {CATALOG}.{SCHEMA}")

## Load the TabFM regression model


In [ ]:
import numpy as np
import pandas as pd
import torch
import mlflow

from tabfm import TabFMRegressor, tabfm_v1_0_0_pytorch as tabfm_v1_0_0
from evaluation import (
    split_xy, regression_metrics, train_baselines_regression,
    log_result, RESULTS_TABLE,
)

mlflow.set_experiment(MLFLOW_EXPERIMENT_NAME)

# TabFM must run on GPU — CPU inference is prohibitively slow (tens of minutes).
device = "cuda" if torch.cuda.is_available() else "cpu"
if device == "cpu":
    print("WARNING: no GPU detected — TabFM inference on CPU is extremely slow. "
          "Attach a GPU cluster (e.g. g5.xlarge).")
tabfm_reg_model = tabfm_v1_0_0.load(model_type="regression", device=device)
print(f"TabFM regression model loaded on {device}.")

## Helper: evaluate one regression task


In [ ]:
def evaluate_regression(table_name, target, task_name, test_size=0.2):
    df = spark.table(table_name).toPandas()
    X_train, X_test, y_train, y_test = split_xy(df, target=target, test_size=test_size)
    n_train, n_features, n_test = len(X_train), X_train.shape[1], len(X_test)

    # n_estimators=1 keeps first-run latency low; raise for a small accuracy bump.
    with mlflow.start_run(run_name=f"{task_name}_tabfm"):
        mlflow.log_params({
            "vendor": "tabfm", "model_type": "TabFMRegressor",
            "task": task_name, "problem_type": "regression",
            "n_features": n_features, "train_samples": n_train, "test_samples": n_test,
        })
        reg = TabFMRegressor(model=tabfm_reg_model, n_estimators=1)
        reg.fit(X_train, y_train)
        y_pred = reg.predict(X_test)
        metrics = regression_metrics(y_test, y_pred)
        mlflow.log_metrics({k: v for k, v in metrics.items() if v is not None})
        log_result(spark, vendor="tabfm", task=task_name, problem_type="regression",
                   model_name="TabFMRegressor", metrics=metrics,
                   n_train=n_train, n_test=n_test, n_features=n_features)
    print(f"[{task_name}] TabFM: rmse={metrics['rmse']:.4f} "
          f"mae={metrics['mae']:.4f} r2={metrics['r2']:.4f}")

    for name, m in train_baselines_regression(X_train, y_train, X_test, y_test).items():
        log_result(spark, vendor="baseline", task=task_name, problem_type="regression",
                   model_name=name, metrics=m,
                   n_train=n_train, n_test=n_test, n_features=n_features)
        print(f"[{task_name}] {name}: rmse={m['rmse']:.4f} r2={m['r2']:.4f}")
    return metrics

## Price Elasticity


In [ ]:
_ = evaluate_regression(
    table_name="price_elasticity_train",
    target="price_elasticity",
    task_name="price_elasticity",
)

## Supplier Lead Time


In [ ]:
_ = evaluate_regression(
    table_name="supplier_lead_time_train",
    target="actual_lead_time_days",
    task_name="supplier_lead_time",
)

## Results


In [ ]:
display(
    spark.table(RESULTS_TABLE)
         .where("problem_type = 'regression'")
         .orderBy("task", "vendor", "model_name")
)